### **Prerequisites**


In [5]:
%pip install dotenv==0.9.9 openai==1.100.0

In [6]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
your_api_key = "<YOUR_API_KEY>" # 여기에 Upstage API 키를 입력하세요.

# .env 파일에 쓰기
!echo "UPSTAGE_API_KEY={your_api_key}" > "/content/drive/My Drive/Colab Notebooks/.env"

In [17]:
from dotenv import load_dotenv
from os import getenv
load_dotenv("/content/drive/My Drive/Colab Notebooks/.env")

UPSTAGE_API_KEY = getenv("UPSTAGE_API_KEY")
if UPSTAGE_API_KEY:
    print("Success API Key Setting!")

Success API Key Setting!


In [18]:
import httpx

async def call_chat_completion(url: str, headers: dict, payload: dict):
    async with httpx.AsyncClient(timeout=30.0) as client:
        response = await client.post(url, headers=headers, json=payload)
        response.raise_for_status()
        data = response.json()

        return data["choices"][0]["message"]["content"]


In [19]:
import asyncio

async def main():
    url = "https://api.upstage.ai/v1/chat/completions"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {UPSTAGE_API_KEY}"
    }
    prompts = [
        "Tell me a joke about cats",
        "What is the capital of France?",
        "Summarize the plot of Inception in 2 sentences."
    ]

    tasks = []
    for prompt in prompts:
        payload = {
            "model": "solar-pro2",
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "stream": False
        }
        tasks.append(call_chat_completion(url, headers, payload))

    results = await asyncio.gather(*tasks)

    for i, res in enumerate(results, 1):
        print(f"--- Response {i} ---")
        print(res)
        print()

# 🔹 실행
await main()

--- Response 1 ---
Why don't cats play poker in the jungle?  
Because there are too many *cheetahs*!  

😸 *Purr-fectly punny, right?* 🎲

--- Response 2 ---
The capital of France is **Paris**.  

Known for its iconic landmarks like the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral, Paris is a major global hub for culture, fashion, gastronomy, and history. Let me know if you'd like more fun facts about the city! 😊

--- Response 3 ---
In *Inception*, a skilled thief named Dom Cobb is offered the chance to have his criminal history erased as payment for infiltrating the mind of a business magnate to plant an idea (an "inception") that could destabilize his empire. The team must navigate multiple dream layers, facing layers of deception, betrayal, and the risk of losing themselves forever in the subconscious.  

*(Condensed to two sentences:)*  
Dom Cobb is tasked with implanting an idea into a CEO's mind to redeem himself, but the dangerous dream-sharing operation threatens to trap

In [20]:
from openai import AsyncOpenAI

client = AsyncOpenAI(api_key=UPSTAGE_API_KEY, base_url="https://api.upstage.ai/v1")

async def chat_completion(prompt: str, model: str = "solar-pro2") -> str:
    """
    비(非)스트리밍 호출 버전. 한 번에 전체 응답을 받아옵니다.
    """
    resp = await client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        stream=False,
    )
    return resp.choices[0].message.content

async def main():
    prompts = [
        "Tell me a joke about cats",
        "What is the capital of France?",
        "Summarize the plot of Inception in 2 sentences."
    ]

    # ✅ 동시에 스트리밍 호출
    tasks = [chat_completion(p) for p in prompts]
    results = await asyncio.gather(*tasks)

    # 결과 정리 출력
    for i, res in enumerate(results, 1):
        print(f"--- Response {i} (collected) ---")
        print(res)
        print()

# Colab/Jupyter에서는 최상단 셀에서 바로 실행 가능
await main()

--- Response 1 (collected) ---
Why don't cats play poker in the jungle?  
Because there are too many *cheetahs*!  

😸 (Pun intended—*cheetahs* sounds like "cheaters!")  

If you need another paw-sitive chuckle, just let me know! 😼

--- Response 2 (collected) ---
The capital of France is **Paris**.  

Would you like to know any interesting facts about Paris? 😊

--- Response 3 (collected) ---
In *Inception*, a thief named Dom Cobb is offered the chance to have his criminal history erased as payment for a seemingly impossible task: "inception," the implantation of an idea into the mind of a C.E.O., using shared dreaming technology that allows entry into the dreams of others. The mission goes perilously awry as layers of dreams within dreams unfold, leaving Cobb struggling to return to reality and reunite with his children.



In [23]:
from openai import OpenAI
import json

client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url="https://api.upstage.ai/v1"
)

user_prompt = """가상의 데이터를 만들려고 합니다.

아래 output_format에 따라 JSON 데이터를 1개만 반환해주세요.
{output_format}
"""

# TODO: output_format을 정의해주세요.
output_format ="""<output_format>
```json
{
    "name": [이름],
    "age": [나이]
    "is_student": [true 또는 false]
}
```
</output_format>"""

messages=[
    {
        "role": "user",
        "content": user_prompt.format(output_format=output_format)
    }
]

response = client.chat.completions.create(
    model="solar-pro2",
    messages=messages,
)

def json_parsing(output_text:str) -> dict:
    output_text = output_text[output_text.index("```json") + len("```json"):].strip()
    output_text = output_text[:output_text.index("```")].strip()
    return json.loads(output_text)


output = response.choices[0].message.content
print("현재 응답의 형식: ", type(output))
print(output)
structured_dictionary = json_parsing(output)
print("구조화된 응답의 형식: ", type(structured_dictionary))
print(structured_dictionary)

현재 응답의 형식:  <class 'str'>
```json
{
    "name": "김민주",
    "age": 24,
    "is_student": false
}
```
구조화된 응답의 형식:  <class 'dict'>
{'name': '김민주', 'age': 24, 'is_student': False}


In [24]:
from openai import OpenAI
import json


messages=[
    {
        "role": "system",
        "content": "You are an expert in information extraction. Extract information from the given HTML representation of image and organize them into a clear and accurate JSON format."
    },
    {
        "role": "user",
        "content": "HTML string: <table id='0' style='font-size:14px'><tr><td>1</td><td>FUTAMI 17 GREEN TEA (CLAS</td><td>12,500</td></tr><tr><td>1</td><td>EGG TART</td><td>13,000</td></tr><tr><td>1</td><td>GRAIN CROQUE MONSIEUR</td><td>17,000</td></tr></table><br><table id='1' style='font-size:18px'><tr><td>TOTAL</td><td>42, 500</td></tr><tr><td>CASH</td><td>50,000</td></tr><tr><td></td><td></td></tr><tr><td>CHANGE</td><td>7 ,500</td></tr></table>\n. Extract the structured data from the HTML string in JSON format."
    }
]

response_format={
    "type": "json_schema",
    "json_schema": {
        "name": "restaurant_receipt",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "menu_items": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "menu_cnt": {
                                "type": "number",
                                "description": "The count of the menu item."
                            },
                            "menu_name": {
                                "type": "string",
                                "description": "The name of the menu item."
                            },
                            "menu_price": {
                                "type": "number",
                                "description": "The price of the menu item."
                            }
                        },
                        "required": ["menu_cnt", "menu_name", "menu_price"],
                    }
                },
                "total_price": {
                    "type": "number",
                    "description": "The total price of the receipt."
                }
            },
            "required": ["menu_items", "total_price"],
        }
    }
}

response = client.chat.completions.create(
    model="solar-pro2",
    messages=messages,
    response_format=response_format
)

structured_output = response.choices[0].message.content
print("현재 응답의 형식: ", type(structured_output))
print(structured_output)
structured_dictionary = json.loads(structured_output)
print("구조화된 응답의 형식: ", type(structured_dictionary))
print(structured_dictionary)

현재 응답의 형식:  <class 'str'>
{
  "menu_items": [
    {
      "menu_cnt": 1,
      "menu_name": "FUTAMI 17 GREEN TEA (CLAS",
      "menu_price": 12500
    },
    {
      "menu_cnt": 1,
      "menu_name": "EGG TART",
      "menu_price": 13000
    },
    {
      "menu_cnt": 1,
      "menu_name": "GRAIN CROQUE MONSIEUR",
      "menu_price": 17000
    }
  ],
  "total_price": 42500
}
구조화된 응답의 형식:  <class 'dict'>
{'menu_items': [{'menu_cnt': 1, 'menu_name': 'FUTAMI 17 GREEN TEA (CLAS', 'menu_price': 12500}, {'menu_cnt': 1, 'menu_name': 'EGG TART', 'menu_price': 13000}, {'menu_cnt': 1, 'menu_name': 'GRAIN CROQUE MONSIEUR', 'menu_price': 17000}], 'total_price': 42500}


In [27]:
GENERATOR_SYSTEM_PROMPT = """당신은 세상의 모든 영화를 꿰뚫고 있는 영화 전문가 '시네마스터'입니다.
사용자의 요청에 맞춰 영화를 추천하는 역할을 맡고 있습니다. 영화는 반드시 하나만 추천합니다.

{rule}

추천할 때는 반드시 영화 제목, 개봉 연도, 그리고 추천 이유를 포함해야 합니다.
"""

# TODO: 아래 두 가지 규칙 중 하나를 선택하여 지시사항을 완성하세요.
# rule = "규칙 A (전문가 모드): 전문적인 용어를 사용하고, 영화의 숨겨진 의미나 감독의 의도를 함께 설명해주세요"
rule = "규칙 B (친구 모드): 친근하고 유머러스한 말투로, 왜 이 영화가 재미있는지 쉽게 설명해주세요."

client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url="https://api.upstage.ai/v1"
)

response = client.chat.completions.create(
    model="solar-pro2",
    messages=[
        {
            "role": "system",
            "content": GENERATOR_SYSTEM_PROMPT.format(rule=rule)
        },
        {
            "role": "user",
            "content": "공포 영화를 추천해줘"
        }
    ],
)

output = response.choices[0].message.content
print(output)

**"허슬러 하우스" (2022)**

> "으악~" 소리 나는 공포보다 **"으헉?!"** 웃음이 터지는 블랙코미디 공포물!  
> 2022년 개봉한 이 영화는 **좀비 아포칼립스 속 히피들의 생존기**를 그렸는데,  
> **"좀비보다 옆집 히피가 더 무섭다"**는 반전 매력에 폭소 & 오싹함을 동시에 선사합니다.  
>  
> 🔪 **추천 이유**:  
> - **"평온한 죽음"**을 추구하는 히피들과 좀비들의 기묘한 동거가 코미디와 공포를 오가게 해요.  
> - **"좀비 영화인데 왜 이렇게 웃기지?"**라는 질문이 나올 정도로 유쾌발랄한 연출!  
> - **실제 70년대 공포영화의 오마주**가 가득해 취향 저격 보장!  
>  
> 🎬 **결론**: 공포 영화를 보다가 **"너희 진짜 웃긴데?"**라는 말이 절로 나온다면, 바로 이 영화 맞습니다!  

> 💀 *주의: 좀비보다 히피들의 **"평화를 빕니다"** 대사에 더 놀랄 수 있음.*  

---  
**"고전 공포물 + 현대 유머"**가 결합된 독특한 맛을 느껴보세요! 👻😆


In [ ]:
# TODO: SYSTEM PROMPT를 작성하세요
STRUCTURED_GENERATOR_SYSTEM_PROMPT = """당신은 세상의 모든 영화를 꿰뚫고 있는 영화 전문가 '시네마스터'입니다.
사용자의 요청에 맞춰 영화를 추천하는 역할을 맡고 있습니다. 영화는 반드시 하나만 추천합니다.

## 1. 입력 형식
[추천받고자 하는 영화 장르]

## 2. 작업 지시
{rule}

## 3. 출력 형식
<output_format>
```json
{{
    "movie_name": [영화 이름],
    "year": [개봉 연도],
    "reason": [추천 이유]
}}
```
</output_format>
"""

response = client.chat.completions.create(
    model="solar-pro2",
    messages=[
        {
            "role": "system",
            "content": STRUCTURED_GENERATOR_SYSTEM_PROMPT.format(rule=rule)
        },
        {
            "role": "user",
            "content": "공포 영화를 추천해줘"
        }
    ],
    temperature=1.0,
)

output = response.choices[0].message.content
structured_dictionary = json_parsing(output)
print("구조화된 응답의 형식: ", type(structured_dictionary))
print(structured_dictionary)

구조화된 응답의 형식:  <class 'dict'>
{'movie_name': 'Get Out', 'year': 2017, 'reason': "『겟 아웃』은 공포물이지만 사회적 메시지도 담겨 있어 더 짜릿해요! 백인만 사는 고급 주택에서 벌어지는 미묘한 인종 긴장감을 '눈속임' 같은 반전으로 풀어내죠. 특히 '미술품 수집'이라는 설정의 은유적 표현이나, 로즈 가족의 어색한 환대는 소름 끼치게 신랄해요. 단순한 공포보다 지적인 공포물을 원하는 당신에게 강추! 감독 조던 필의 유머 감각도 숨어 있어 공포 뒤에 웃을 순간도 있답니다."}


In [29]:
JUDGE_SYSTEM_PROMPT="""당신의 역할은 모델 답변 자동 평가자입니다. 입력 프롬프트와 모델 답변을 보고, 평가 기준에 따라 모델 답변을 평가합니다.

## 1. 입력 형식
    - 입력 프롬프트: [instruction]
    - 모델 답변: [output]
    - 평가 기준: [criteria]

## 2. 작업 지시
    - [instruction]에 따른 모델 결과물인 [output]을 평가합니다.
    - [output]은 [criteria]를 충족하는지 평가합니다.

## 3. 채점 원칙 (각 기준별 1–5점, 정수만)
    - 5점 (탁월): 기준을 완전히 충족. 오류·누락 없음. 구체적이고 실행가능.
	- 4점 (우수): 대체로 충족. 사소한 흠만 있음(정확성·구체성·형식 등에서 경미한 누락).
	- 3점 (보통): 핵심은 맞지만 눈에 띄는 약점 존재(누락, 모호함, 근거 부족 등).
    - 2점 (미흡): 중요한 요구를 여러 곳에서 놓침 또는 오류/비논리 다수.
	- 1점 (부적합): 전반적으로 요청과 어긋남, 의미있는 도움/근거 없음, 안전·정책 위반 가능성.

## 3. 출력 형식 (엄격 준수)
	- "score"는 1–5점의 정수로 평가한다.
	- "reason"는 한국어 1–3문장으로 평가한다. 구체적이고 실행 가능하게 작성한다.
	- 출력 형식은 JSON 형식인 <output_format>을 준수한다.

<output_format>
```json
{{
    "score": [모델의 답변 평가 점수],
    "comment": [평가 주석]
}}
```
"""

USER_PROMPT = """- 입력 프롬프트: {instruction}
- 모델 답변: {output}
- 평가 기준: {criteria}
"""

instruction = GENERATOR_SYSTEM_PROMPT.format(rule=rule)
# TODO: 기준을 정의해주세요
criteria = "요청의 충실도"

print(USER_PROMPT.format(instruction=instruction, output=output, criteria=criteria))

messages = [
        {
            "role": "system",
            "content": JUDGE_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": USER_PROMPT.format(instruction=instruction, output=output, criteria=criteria)
        }
]

response = client.chat.completions.create(
    model="solar-pro2",
    messages=messages
)

llm_as_judge_output = response.choices[0].message.content
llm_as_judge_structured_dictionary = json_parsing(llm_as_judge_output)
print("구조화된 응답의 형식: ", type(llm_as_judge_structured_dictionary))
print(llm_as_judge_structured_dictionary)

- 입력 프롬프트: 당신은 세상의 모든 영화를 꿰뚫고 있는 영화 전문가 '시네마스터'입니다.
사용자의 요청에 맞춰 영화를 추천하는 역할을 맡고 있습니다. 영화는 반드시 하나만 추천합니다.

규칙 B (친구 모드): 친근하고 유머러스한 말투로, 왜 이 영화가 재미있는지 쉽게 설명해주세요.

추천할 때는 반드시 영화 제목, 개봉 연도, 그리고 추천 이유를 포함해야 합니다.

- 모델 답변: <output_format>
```json
{
    "movie_name": "Get Out",
    "year": 2017,
    "reason": "『겟 아웃』은 공포물이지만 사회적 메시지도 담겨 있어 더 짜릿해요! 백인만 사는 고급 주택에서 벌어지는 미묘한 인종 긴장감을 '눈속임' 같은 반전으로 풀어내죠. 특히 '미술품 수집'이라는 설정의 은유적 표현이나, 로즈 가족의 어색한 환대는 소름 끼치게 신랄해요. 단순한 공포보다 지적인 공포물을 원하는 당신에게 강추! 감독 조던 필의 유머 감각도 숨어 있어 공포 뒤에 웃을 순간도 있답니다."
}
```
</output_format>
- 평가 기준: 요청의 충실도

구조화된 응답의 형식:  <class 'dict'>
{'score': 5, 'comment': "모든 요구사항을 완벽하게 충족했습니다. 영화 제목, 개봉 연도, 추천 이유를 명시했으며, 규칙 B의 '친구 모드'에 맞춰 유머와 친근감을 담은 설명을 제공했습니다. 공포 장르의 특성과 사회적 메시지를 결합해 흥미롭게 서술했고, 감독에 대한 언급으로 신뢰성을 높였습니다. 형식적으로도 오류가 없습니다."}
